In [ ]:
# Cellule 1: Correction - les images sont déjà dans les dossiers
import os
import glob
import shutil

# Les images sont déjà dans ces dossiers
source_dir_1 = '/content/skin_data/HAM10000_images_part_1'
source_dir_2 = '/content/skin_data/HAM10000_images_part_2'

# Créer le dossier cible
os.makedirs('/content/skin_data/images', exist_ok=True)

# Copier les images depuis les dossiers existants
print("Déplacement des images...")

if os.path.exists(source_dir_1) and os.path.isdir(source_dir_1):
    for img in glob.glob(f'{source_dir_1}/*.jpg'):
        shutil.copy(img, '/content/skin_data/images/')
    print(f"✓ Images copiées depuis {source_dir_1}")

if os.path.exists(source_dir_2) and os.path.isdir(source_dir_2):
    for img in glob.glob(f'{source_dir_2}/*.jpg'):
        shutil.copy(img, '/content/skin_data/images/')
    print(f"✓ Images copiées depuis {source_dir_2}")

# Vérification
images = glob.glob('/content/skin_data/images/*.jpg')
print(f"\n✅ {len(images)} images disponibles")

# Afficher les 5 premières
if len(images) > 0:
    print("\nExemples d'images:")
    for img in images[:3]:
        print(f"  - {os.path.basename(img)}")

In [2]:
# Cellule 1: Imports et configuration améliorée
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# Configuration
plt.style.use('seaborn-v0_8')
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Using device: cpu


In [3]:
# Cellule 2: Dataset personnalisé avec meilleures augmentations
class SkinLesionDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'image_id'] + '.jpg'
        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = cv2.imread(img_path)
            if image is None:
                raise FileNotFoundError(f"Image non trouvée: {img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except Exception as e:
            print(f"Erreur chargement {img_path}: {e}")
            # Image de remplacement (noire)
            image = np.zeros((224, 224, 3), dtype=np.uint8)

        label = self.df.loc[idx, 'dx']
        label_idx = self.df.loc[idx, 'label_idx']

        if self.transform:
            image = self.transform(image)

        return image, label_idx

# Augmentations améliorées pour l'entraînement
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Augmentations légères pour la validation
val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# Cellule 3: Chargement et préparation des données
metadata = pd.read_csv('/content/skin_data/HAM10000_metadata.csv')

# Vérification des images existantes
existing_images = set([os.path.basename(f).replace('.jpg', '')
                       for f in glob.glob('/content/skin_data/images/*.jpg')])
metadata['exists'] = metadata['image_id'].isin(existing_images)
metadata = metadata[metadata['exists']].reset_index(drop=True)

# Encodage des labels
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
metadata['label_idx'] = label_encoder.fit_transform(metadata['dx'])

# Affichage de la distribution des classes
class_names = label_encoder.classes_
class_counts = metadata['dx'].value_counts()

print("Distribution des classes:")
for dx, count in class_counts.items():
    print(f"  {dx}: {count} ({count/len(metadata)*100:.1f}%)")

# Division train/validation (80/20) avec stratification
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    metadata,
    test_size=0.2,
    stratify=metadata['dx'],
    random_state=42
)

print(f"\nTrain set: {len(train_df)} images")
print(f"Validation set: {len(val_df)} images")

# Création des datasets
train_dataset = SkinLesionDataset(train_df, '/content/skin_data/images', transform=train_transforms)
val_dataset = SkinLesionDataset(val_df, '/content/skin_data/images', transform=val_transforms)

# Calcul des poids pour l'échantillonnage pondéré (gestion du déséquilibre)
class_counts_train = train_df['dx'].value_counts()
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['dx']),
    y=train_df['dx']
)
class_weights_dict = dict(zip(np.unique(train_df['dx']), class_weights))
sample_weights = [class_weights_dict[dx] for dx in train_df['dx']]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Poids pour la fonction de perte
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\nPoids des classes: {class_weights_tensor.cpu().numpy()}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/skin_data/HAM10000_metadata.csv'

In [ ]:
# Cellule 4: Modèle ResNet50 avec tête de classification améliorée
class ImprovedResNet50(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.3, freeze_layers=False):
        super(ImprovedResNet50, self).__init__()

        # Chargement du modèle pré-entraîné
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        # Gel des couches si demandé
        if freeze_layers:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Remplacer la dernière couche par une tête plus complexe
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()  # Enlève la fc originale

        # Nouvelle tête de classification
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate/2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

    def unfreeze_layers(self, num_layers=3):
        """Dégèle les derniers blocs pour le fine-tuning"""
        # Récupère les noms des paramètres
        layers = list(self.backbone.named_parameters())

        # Dégèle les derniers num_layers blocs
        for i, (name, param) in enumerate(layers[-num_layers*10:]):
            param.requires_grad = True
        print(f"✓ {num_layers} derniers blocs dégelés")

In [ ]:
# Cellule 5: Fonction d'entraînement avec métriques avancées
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Training'):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping pour éviter l'explosion des gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = balanced_accuracy_score(all_labels, all_preds)  # Balanced accuracy!

    return epoch_loss, epoch_acc, all_preds, all_labels

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = balanced_accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:
# Cellule 6: Entraînement principal avec callbacks
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs=8, patience=7, model_name="model", device='cuda'):

    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f"\n{'='*50}")
    print(f"Début de l'entraînement de {model_name}")
    print(f"{'='*50}")

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")

        # Entraînement
        train_loss, train_acc, _, _ = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_preds, val_labels = validate_epoch(model, val_loader, criterion, device)

        # Mise à jour du scheduler
        if scheduler:
            if isinstance(scheduler, ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        # Sauvegarde de l'historique
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f}, Train Acc (balanced): {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc (balanced): {val_acc:.4f}")

        # Early stopping et sauvegarde du meilleur modèle
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, f'{model_name}_best.pth')
            print(f"✓ Nouveau best {model_name}! Acc: {val_acc:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⚠️ Early stopping à l'époque {epoch+1}")
                break

    # Affichage du résumé
    print(f"\n{'='*50}")
    print(f"Résumé de l'entraînement de {model_name}")
    print(f"{'='*50}")
    print(f"Meilleure époque: {best_epoch}")
    print(f"Meilleure accuracy (balanced): {best_val_acc:.4f}")

    # Chargement du meilleur modèle
    checkpoint = torch.load(f'{model_name}_best.pth')
    model.load_state_dict(checkpoint['model_state_dict'])

    return model, history, best_val_acc

# Création du modèle
model = ImprovedResNet50(num_classes=7, dropout_rate=0.3, freeze_layers=False).to(device)

# Dégeler les dernières couches (fine-tuning progressif)
model.unfreeze_layers(num_layers=3)

# Optimiseur avec weight decay
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Scheduler avec ReduceLROnPlateau
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# Fonction de perte avec poids
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Entraînement
num_epochs = 8
model, history, best_acc = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    num_epochs=num_epochs, patience=8, model_name="improved_resnet50", device=device
)

In [ ]:
# Cellule 7: Visualisation des courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Courbes de Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc (balanced)', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc (balanced)', marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title("Courbes d'Accuracy (Balanced)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Cellule 8: Évaluation détaillée du meilleur modèle
def evaluate_model(model, loader, criterion, device, class_names):
    model.eval()
    all_preds = []
    all_labels = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluation'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    standard_acc = np.mean(np.array(all_preds) == np.array(all_labels))

    print(f"\n{'='*50}")
    print("Évaluation Finale du Modèle")
    print(f"{'='*50}")
    print(f"Loss moyenne: {avg_loss:.4f}")
    print(f"Accuracy standard: {standard_acc:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")

    # Rapport de classification
    print(f"\n{'='*50}")
    print("Classification Report")
    print(f"{'='*50}")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Matrice de confusion
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Prédictions')
    plt.ylabel('Vérité terrain')
    plt.title('Matrice de Confusion')
    plt.tight_layout()
    plt.show()

    return balanced_acc, standard_acc, all_preds, all_labels

# Évaluation
val_balanced_acc, val_standard_acc, _, _ = evaluate_model(
    model, val_loader, criterion, device, class_names
)

In [ ]:
# Cellule 9: Fonction de prédiction pour une nouvelle image
def predict_image(model, image_path, transform, device, class_names):
    """
    Prédit la classe d'une image de lésion cutanée
    """
    # Chargement de l'image
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Image non trouvée: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Transformation
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Prédiction
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        pred_class = torch.argmax(probabilities, dim=1).item()

    # Résultats
    confidence = probabilities[0][pred_class].item()
    predicted_label = class_names[pred_class]

    # Top-3 prédictions
    top3_probs, top3_indices = torch.topk(probabilities[0], 3)
    top3_predictions = [(class_names[idx], prob.item()) for idx, prob in zip(top3_indices, top3_probs)]

    # Affichage
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title(f"Prédiction: {predicted_label}\nConfiance: {confidence:.2%}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    classes = [p[0] for p in top3_predictions]
    probs = [p[1] for p in top3_predictions]
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(classes))]
    plt.barh(classes, probs, color=colors)
    plt.xlabel('Probabilité')
    plt.title('Top 3 Prédictions')
    plt.xlim(0, 1)

    plt.tight_layout()
    plt.show()

    return predicted_label, confidence, top3_predictions

# Exemple d'utilisation (à décommenter)
# image_path = "/content/skin_data/images/ISIC_0031861.jpg"
# predicted_label, confidence, top3 = predict_image(model, image_path, val_transforms, device, class_names)
# print(f"Prédiction: {predicted_label} (confiance: {confidence:.2%})")
# print("Top 3:", top3)

In [ ]:
# Cellule 10: Sauvegarde des résultats
import pickle

# Sauvegarde des résultats
results = {
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'val_balanced_acc': val_balanced_acc,
    'val_standard_acc': val_standard_acc,
    'history': history,
    'class_weights': class_weights_tensor.cpu().numpy(),
    'train_transforms': train_transforms,
    'val_transforms': val_transforms
}

torch.save(results, 'improved_skin_lesion_model.pth')
print("✓ Modèle sauvegardé dans 'improved_skin_lesion_model.pth'")

# Sauvegarde des métriques
with open('training_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print("✓ Historique d'entraînement sauvegardé")

# Résumé final
print(f"\n{'='*50}")
print("RÉSUMÉ FINAL")
print(f"{'='*50}")
print(f"Classes: {list(class_names)}")
print(f"Accuracy standard: {val_standard_acc:.4f} ({val_standard_acc*100:.2f}%)")
print(f"Balanced Accuracy: {val_balanced_acc:.4f} ({val_balanced_acc*100:.2f}%)")